In [2]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score


# ============================================================
# 1. LOAD PLAYER DATA
# ============================================================

df = pd.read_csv("player_stats_engineered.csv")

print("Player data shape:", df.shape)
print("Player columns:")
print(df.columns.tolist())

df.head()


# ============================================================
# 2. PREPARE PLAYER DATA
# ============================================================

# Convert game_date to datetime
df["game_date"] = pd.to_datetime(df["game_date"], errors="coerce")

# Clean location just in case it has values like "Location.HOME"
df["location_clean"] = (
    df["location"]
    .astype(str)
    .str.replace("Location.", "", regex=False)
    .str.upper()
)

# Turn home/away into a numeric feature
df["is_home"] = (df["location_clean"] == "HOME").astype(int)

# Features known before the game
feature_cols = [
    "rolling_pts_5",
    "rolling_reb_5",
    "rolling_ast_5",
    "rolling_min_5",
    "rolling_fg_pct_5",
    "rolling_3p_pct_5",
    "rolling_pts_10",
    "rolling_reb_10",
    "rolling_ast_10",
    "rolling_min_10",
    "rolling_fg_pct_10",
    "rolling_3p_pct_10",
    "home_away_pts_avg",
    "home_away_reb_avg",
    "home_away_ast_avg",
    "rest_days",
    "is_back_to_back",
    "is_home",
]

# Player stat targets
targets = {
    "points": "points",
    "rebounds": "total_rebounds",
    "assists": "assists",
}

# Check for missing columns
needed_cols = (
    ["game_date", "slug", "name", "team", "opponent", "location"]
    + feature_cols
    + list(targets.values())
)

missing_cols = [col for col in needed_cols if col not in df.columns]

print("Missing player columns:", missing_cols)

if missing_cols:
    raise ValueError(f"Missing columns: {missing_cols}")

# Sort by date for time-based split
df = df.dropna(subset=["game_date"]).sort_values("game_date").reset_index(drop=True)

print("Player date range:", df["game_date"].min(), "to", df["game_date"].max())
print("Player rows:", len(df))


# ============================================================
# 3. PLAYER TRAIN / TEST SPLIT
# ============================================================

player_unique_dates = sorted(df["game_date"].dropna().unique())

player_cutoff_index = int(len(player_unique_dates) * 0.75)
player_cutoff_date = player_unique_dates[player_cutoff_index]

player_train_df = df[df["game_date"] < player_cutoff_date].copy()
player_test_df = df[df["game_date"] >= player_cutoff_date].copy()

print("Player cutoff date:", pd.to_datetime(player_cutoff_date).date())
print("Player train rows:", player_train_df.shape)
print("Player test rows:", player_test_df.shape)


# ============================================================
# 4. BUILD LINEAR REGRESSION PLAYER MODEL
# ============================================================

def build_linear_model():
    """
    Linear Regression model for player stat prediction.
    Predicts exact points, rebounds, and assists.
    """
    model = Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), feature_cols)
        ])),
        ("model", LinearRegression())
    ])

    return model


# ============================================================
# 5. TRAIN PLAYER STAT MODELS
# ============================================================

trained_models = {}

for target_name, target_col in targets.items():
    print("\n==============================")
    print(f"Training Linear Regression for: {target_name}")
    print("==============================")

    train_target_df = player_train_df.dropna(subset=[target_col]).copy()

    X_train = train_target_df[feature_cols]
    y_train = train_target_df[target_col]

    model = build_linear_model()
    model.fit(X_train, y_train)

    trained_models[(target_name, "linear_regression")] = model

    print(f"{target_name} model trained.")


# ============================================================
# 6. LOAD TEAM DATA
# ============================================================

team_df = pd.read_csv("team_game_stats_engineered.csv")

print("Team data shape:", team_df.shape)
print("Team columns:")
print(team_df.columns.tolist())

team_df.head()


# ============================================================
# 7. PREPARE TEAM MATCHUP DATA
# ============================================================

team_df["game_date"] = pd.to_datetime(team_df["game_date"], errors="coerce")

team_df["location_clean"] = (
    team_df["location"]
    .astype(str)
    .str.replace("Location.", "", regex=False)
    .str.upper()
)

team_df = (
    team_df
    .dropna(subset=["game_date"])
    .sort_values(["team", "game_date"])
    .reset_index(drop=True)
)

home_rows = team_df[team_df["location_clean"] == "HOME"].copy()
away_rows = team_df[team_df["location_clean"] == "AWAY"].copy()

home_rows = home_rows.rename(columns={
    col: f"home_{col}" for col in home_rows.columns if col != "game_date"
})

away_rows = away_rows.rename(columns={
    col: f"away_{col}" for col in away_rows.columns if col != "game_date"
})

matchup_df = home_rows.merge(
    away_rows,
    left_on=["game_date", "home_team", "home_opponent"],
    right_on=["game_date", "away_opponent", "away_team"],
    how="inner"
)

# Make sure home_team_win is numeric 0/1
if matchup_df["home_won"].dtype == bool:
    matchup_df["home_team_win"] = matchup_df["home_won"].astype(int)
else:
    matchup_df["home_team_win"] = (
        matchup_df["home_won"]
        .astype(str)
        .str.upper()
        .map({
            "TRUE": 1,
            "FALSE": 0,
            "1": 1,
            "0": 0,
            "W": 1,
            "L": 0,
            "WIN": 1,
            "LOSS": 0,
        })
    )

print("Matchup shape:", matchup_df.shape)
matchup_df[["game_date", "home_team", "away_team", "home_team_win"]].head()


# ============================================================
# 8. BUILD WIN MODEL FEATURES
# ============================================================

base_team_features = [
    "rolling_pts_scored_5",
    "rolling_pts_allowed_5",
    "rolling_win_pct_5",
    "rolling_pts_scored_10",
    "rolling_pts_allowed_10",
    "rolling_win_pct_10",
    "home_away_pts_scored_avg",
    "home_away_pts_allowed_avg",
    "rest_days",
    "is_back_to_back",
]

win_feature_cols = []

for feature in base_team_features:
    home_col = f"home_{feature}"
    away_col = f"away_{feature}"

    if home_col in matchup_df.columns and away_col in matchup_df.columns:
        diff_col = f"diff_{feature}"
        matchup_df[diff_col] = matchup_df[home_col] - matchup_df[away_col]
        win_feature_cols.extend([home_col, away_col, diff_col])

print("Number of win features:", len(win_feature_cols))
print(win_feature_cols)


# ============================================================
# 9. WIN TRAIN / TEST SPLIT
# ============================================================

matchup_df = matchup_df.sort_values("game_date").reset_index(drop=True)

win_unique_dates = sorted(matchup_df["game_date"].dropna().unique())

win_cutoff_index = int(len(win_unique_dates) * 0.75)
win_cutoff_date = win_unique_dates[win_cutoff_index]

win_train_df = matchup_df[matchup_df["game_date"] < win_cutoff_date].copy()
win_test_df = matchup_df[matchup_df["game_date"] >= win_cutoff_date].copy()

print("Win cutoff date:", pd.to_datetime(win_cutoff_date).date())
print("Win train shape:", win_train_df.shape)
print("Win test shape:", win_test_df.shape)


# ============================================================
# 10. BUILD LOGISTIC REGRESSION WIN MODEL
# ============================================================

def build_logistic_model():
    """
    Logistic Regression model for team win prediction.
    Predicts home team win probability.
    """
    model = Pipeline([
        ("preprocess", ColumnTransformer([
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), win_feature_cols)
        ])),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ])

    return model


# ============================================================
# 11. TRAIN WIN MODEL
# ============================================================

X_train = win_train_df[win_feature_cols]
y_train = win_train_df["home_team_win"]

X_test = win_test_df[win_feature_cols]
y_test = win_test_df["home_team_win"]

trained_win_models = {}

print("\n==============================")
print("Training Logistic Regression win model")
print("==============================")

logistic_model = build_logistic_model()
logistic_model.fit(X_train, y_train)

win_probs = logistic_model.predict_proba(X_test)[:, 1]
win_preds = (win_probs >= 0.5).astype(int)

win_accuracy = accuracy_score(y_test, win_preds)

trained_win_models["logistic_regression"] = logistic_model

print("Logistic Regression win model trained.")
print("Accuracy:", win_accuracy)


# ============================================================
# 12. WIN PROBABILITY PREDICTION FUNCTION
# ============================================================

def predict_existing_matchup_win_probability(
    game_date,
    home_team,
    away_team,
    model_name="logistic_regression"
):
    """
    Predicts win probability for an existing matchup in the dataset.
    Uses Logistic Regression.
    """

    game_date = pd.to_datetime(game_date)

    game_row = matchup_df[
        (matchup_df["game_date"] == game_date)
        & (matchup_df["home_team"] == home_team)
        & (matchup_df["away_team"] == away_team)
    ].copy()

    if game_row.empty:
        print("No matchup found.")
        print("Check the date/team spelling.")
        return None

    model = trained_win_models[model_name]

    home_prob = model.predict_proba(game_row[win_feature_cols])[:, 1][0]
    away_prob = 1 - home_prob

    predicted_winner = home_team if home_prob >= 0.5 else away_team
    actual_winner = home_team if game_row["home_team_win"].iloc[0] == 1 else away_team

    result = {
        "game_date": game_date.date(),
        "home_team": home_team,
        "away_team": away_team,
        "home_win_probability": round(float(home_prob), 4),
        "away_win_probability": round(float(away_prob), 4),
        "predicted_winner": predicted_winner,
        "actual_winner": actual_winner,
        "model_used": "logistic_regression"
    }

    return result


# ============================================================
# 13. PLAYER STAT PREDICTION FUNCTION
# ============================================================

def predict_player_stats_for_matchup(
    game_date,
    home_team,
    away_team,
    model_name="linear_regression",
    min_minutes=10
):
    """
    Predicts player points, rebounds, and assists for an existing matchup.
    Uses Linear Regression.
    """

    game_date = pd.to_datetime(game_date)

    matchup_players = df[
        (df["game_date"] == game_date)
        &
        (
            ((df["team"] == home_team) & (df["opponent"] == away_team))
            |
            ((df["team"] == away_team) & (df["opponent"] == home_team))
        )
    ].copy()

    if matchup_players.empty:
        print("No player rows found for this matchup.")
        print("Check the date and team names.")
        return None

    result = matchup_players[
        [
            "game_date",
            "slug",
            "name",
            "team",
            "opponent",
            "location",
            "rolling_min_5",
            "points",
            "total_rebounds",
            "assists",
        ]
    ].copy()

    points_model = trained_models[("points", model_name)]
    rebounds_model = trained_models[("rebounds", model_name)]
    assists_model = trained_models[("assists", model_name)]

    result["predicted_points"] = points_model.predict(matchup_players[feature_cols])
    result["predicted_rebounds"] = rebounds_model.predict(matchup_players[feature_cols])
    result["predicted_assists"] = assists_model.predict(matchup_players[feature_cols])

    result = result.rename(columns={
        "points": "actual_points",
        "total_rebounds": "actual_rebounds",
        "assists": "actual_assists",
    })

    # Keep likely rotation players only
    result = result[result["rolling_min_5"].fillna(0) >= min_minutes]

    # Round values for clean display
    result["predicted_points"] = result["predicted_points"].round(1)
    result["predicted_rebounds"] = result["predicted_rebounds"].round(1)
    result["predicted_assists"] = result["predicted_assists"].round(1)

    result = result.sort_values("predicted_points", ascending=False).reset_index(drop=True)

    return result


# ============================================================
# 14. FULL MATCHUP REPORT FUNCTION
# ============================================================

def predict_full_matchup_report(game_date, home_team, away_team):
    """
    Full model-only report:
    - Logistic Regression win probability
    - Linear Regression player stat-line projections
    """

    print("===================================")
    print("NBA Predictive Modeling Report")
    print("===================================")
    print(f"Matchup: {away_team} at {home_team}")
    print(f"Date: {game_date}")
    print()

    # Win probability using Logistic Regression
    win_result = predict_existing_matchup_win_probability(
        game_date=game_date,
        home_team=home_team,
        away_team=away_team,
        model_name="logistic_regression"
    )

    if win_result is not None:
        print("Win Prediction Model: Logistic Regression")
        print("-----------------------------------------")
        print(f"{home_team}: {win_result['home_win_probability'] * 100:.1f}%")
        print(f"{away_team}: {win_result['away_win_probability'] * 100:.1f}%")
        print(f"Predicted winner: {win_result['predicted_winner']}")
        print(f"Actual winner: {win_result['actual_winner']}")
        print()

    # Player stat lines using Linear Regression
    player_result = predict_player_stats_for_matchup(
        game_date=game_date,
        home_team=home_team,
        away_team=away_team,
        model_name="linear_regression",
        min_minutes=10
    )

    if player_result is not None:
        print("Player Stat-Line Model: Linear Regression")
        print("-----------------------------------------")

        display_cols = [
            "name",
            "team",
            "predicted_points",
            "predicted_rebounds",
            "predicted_assists",
            "actual_points",
            "actual_rebounds",
            "actual_assists",
        ]

        display(player_result[display_cols])

    return win_result, player_result


# ============================================================
# 15. TEST ON ONE TEST-SPLIT MATCHUP
# ============================================================

win_result, player_result = predict_full_matchup_report(
    game_date="2026-04-07",
    home_team="BOSTON_CELTICS",
    away_team="CHARLOTTE_HORNETS"
)

Player data shape: (26669, 44)
Player columns:
['id', 'slug', 'name', 'team', 'opponent', 'location', 'outcome', 'game_date', 'seconds_played', 'made_field_goals', 'attempted_field_goals', 'made_three_point_field_goals', 'attempted_three_point_field_goals', 'made_free_throws', 'attempted_free_throws', 'offensive_rebounds', 'defensive_rebounds', 'assists', 'steals', 'blocks', 'turnovers', 'personal_fouls', 'plus_minus', 'game_score', 'points', 'total_rebounds', 'minutes', 'rolling_pts_5', 'rolling_reb_5', 'rolling_ast_5', 'rolling_min_5', 'rolling_fg_pct_5', 'rolling_3p_pct_5', 'rolling_pts_10', 'rolling_reb_10', 'rolling_ast_10', 'rolling_min_10', 'rolling_fg_pct_10', 'rolling_3p_pct_10', 'home_away_pts_avg', 'home_away_reb_avg', 'home_away_ast_avg', 'rest_days', 'is_back_to_back']
Missing player columns: []
Player date range: 2025-10-21 00:00:00 to 2026-04-12 00:00:00
Player rows: 26669
Player cutoff date: 2026-03-01
Player train rows: (19438, 46)
Player test rows: (7231, 46)

Trainin

,name,team,predicted_points,predicted_rebounds,predicted_assists,actual_points,actual_rebounds,actual_assists
0,Jaylen Brown,BOSTON_CELTICS,28.0,6.2,5.6,35,9,3
1,Jayson Tatum,BOSTON_CELTICS,21.8,10.1,5.5,23,5,4
2,LaMelo Ball,CHARLOTTE_HORNETS,20.2,4.5,6.7,36,5,6
3,Brandon Miller,CHARLOTTE_HORNETS,18.3,4.9,3.6,20,0,2
4,Payton Pritchard,BOSTON_CELTICS,17.0,3.4,4.0,12,4,3
5,Kon Knueppel,CHARLOTTE_HORNETS,15.9,5.2,3.3,13,5,5
6,Miles Bridges,CHARLOTTE_HORNETS,15.7,5.5,2.8,13,12,4
7,Derrick White,BOSTON_CELTICS,13.9,4.3,4.2,12,2,3
8,Neemias Queta,BOSTON_CELTICS,12.5,8.6,2.7,12,5,3
9,Nikola Vučević,BOSTON_CELTICS,9.9,5.9,1.9,2,7,2
